In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(f"./../")

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RXGate
from src.mcrx_simplifier import MCRXCascadeSimplifier
from src.misc import (
    get_state, 
    compare_quantum_states, 
    multi_crx,
    count_gates_direct_transpilation
)

In [ ]:
def run_example_with_direct_transpilation(qc_original, description=""):
    """
    Run optimization example using ONLY direct transpilation (no estimation, no fallback).
    
    Args:
        description: Description of the test case
        circuit_creation_func: Function that creates the test circuit
    """
    print(f"\n{'='*80}")
    print(f"EXAMPLE: {description}")
    print(f"{'='*80}")
    
    try:

        print(f"\n--- Original Circuit ---")
        print(qc_original.draw())
        print(f"Original gates: {len(qc_original.data)}")
        print(f"Original depth: {qc_original.depth()}")

        # Transpile original circuit using direct transpilation only
        cx_orig, u3_orig, success_orig, method_orig, transpiled_orig = count_gates_direct_transpilation(
            qc_original
        )

        if not success_orig:
            print(f"❌ Direct transpilation failed for original circuit")
            return {'equivalence': '❌ DIRECT TRANSPILATION FAILED'}

        print(f"Gate counts ({method_orig}) - CX: {cx_orig}, U3: {u3_orig}")
        print(f"Transpiled circuit depth: {transpiled_orig.depth()}")
        print(f"Transpiled gate types: {list(transpiled_orig.count_ops().keys())}")

        # Optimization analysis
        print(f"\n--- Optimization Analysis ---")
        simplifier = MCRXCascadeSimplifier(tolerance=1e-10, verbose=True)
        try:
            analysis = simplifier.analyze_patterns(qc_original)
            if analysis.get('status') == 'success':
                potential = analysis.get('potential_reduction', 0)
                reduction_pct = (potential / len(qc_original.data)) * 100 if len(qc_original.data) > 0 else 0
                print(f"Optimization potential: {reduction_pct:.1f}%")
                print(f"XOR pairs found: {len(analysis.get('xor_pairs', []))}")
                print(f"Can optimize: {analysis.get('can_optimize', False)}")
            else:
                print(f"Analysis failed: {analysis.get('error', 'Unknown error')}")
        except Exception as e:
            print(f"Pattern analysis failed: {e}")

        # Apply simplification
        print(f"\n--- Applying Optimization ---")
        try:
            result = simplifier.simplify(qc_original)
            if isinstance(result, tuple):
                qc_optimized, optimization_info = result
            else:
                qc_optimized = result
                optimization_info = {}

            print(f"✓ Optimization completed")
            print(f"Optimized gates: {len(qc_optimized.data)}")
            print(f"Optimized depth: {qc_optimized.depth()}")
        except Exception as e:
            print(f"❌ Optimization failed: {e}")
            return {'equivalence': "✗ OPTIMIZATION FAILED"}

        # Transpile optimized circuit
        cx_opt, u3_opt, success_opt, method_opt, transpiled_opt = count_gates_direct_transpilation(
            qc_optimized
        )

        if not success_opt:
            print(f"❌ Direct transpilation failed for optimized circuit")
            return {'equivalence': '❌ OPTIMIZED TRANSPILATION FAILED'}

        print(f"Gate counts ({method_opt}) - CX: {cx_opt}, U3: {u3_opt}")
        print(f"Transpiled optimized depth: {transpiled_opt.depth()}")

        # State comparison
        print(f"\n--- State Equivalence Verification ---")
        try:
            state_orig = get_state(qc_original)
            state_opt = get_state(qc_optimized)
            results = compare_quantum_states(state_orig, state_opt, tolerance=1e-10, verbose=True)

            equivalence = "✓ EQUIVALENT" if results['amplitudes_match'] else "✗ NOT EQUIVALENT"
        except Exception as e:
            print(f"State verification failed: {e}")
            equivalence = "? VERIFICATION FAILED"
            results = {'fidelity': 0.0}

        # Performance summary
        print(f"\n--- Performance Summary ---")
        cx_red = max(0, cx_orig - cx_opt)
        u3_red = max(0, u3_orig - u3_opt)
        total_red = cx_red + u3_red

        cx_pct = (cx_red / max(1, cx_orig)) * 100
        u3_pct = (u3_red / max(1, u3_orig)) * 100
        total_pct = (total_red / max(1, cx_orig + u3_orig)) * 100

        depth_red = max(0, qc_original.depth() - qc_optimized.depth())
        depth_pct = (depth_red / max(1, qc_original.depth())) * 100

        print(f"CX gate reduction: {cx_red} ({cx_pct:.1f}%)")
        print(f"U3 gate reduction: {u3_red} ({u3_pct:.1f}%)")
        print(f"Depth reduction: {depth_red} ({depth_pct:.1f}%)")
        print(f"Fidelity: {results.get('fidelity', 0.0):.10f}")

        return {
            'equivalence': equivalence,
            'gate_reduction': total_pct,
            'depth_reduction': depth_pct,
            'cx_reduction': cx_pct,
            'u3_reduction': u3_pct,
            'analysis_method': f"{method_orig}/{method_opt}",
            'optimization_info': optimization_info,
            'transpiled_counts': {
                'original': {'cx': cx_orig, 'u3': u3_orig},
                'optimized': {'cx': cx_opt, 'u3': u3_opt}
            },
            'transpilation_success': True
        }

    except Exception as e:
        print(f"Critical error: {e}")
        return {
            'equivalence': "✗ CRITICAL FAILURE",
            'transpilation_success': False
        }


In [3]:
# Example 1: Fully Disjoint Control Patterns
theta = np.pi / 4
qc = QuantumCircuit(3)
rx1 = multi_crx(theta, '10')
rx2 = multi_crx(theta, '01')
qc.append(rx1, [0, 1, 2])
qc.append(rx2, [0, 1, 2])


result1 = run_example_with_direct_transpilation(qc, "Fully Disjoint Control Patterns ['10', '01']")


EXAMPLE: Fully Disjoint Control Patterns ['10', '01']

--- Original Circuit ---
                           
q_0: ─────■──────────o─────
          │          │     
q_1: ─────o──────────■─────
     ┌────┴────┐┌────┴────┐
q_2: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Direct-Transpile) - CX: 16, U3: 17
Transpiled circuit depth: 25
Transpiled gate types: ['cx', 'u', 'u3', 'x', 'p']

--- Optimization Analysis ---
Optimization potential: 0.0%
XOR pairs found: 1
Can optimize: True

--- Applying Optimization ---
🔬 Starting  pattern reading MCRX simplification...
✓ Circuit validated: target=2, angle=0.7854, controls=2
✓ Extracted 2 patterns:
  ControlPattern('10', coeff=1, qubits=[0, 1])
  ControlPattern('01', coeff=1, qubits=[0, 1])
✓ Original Boolean: (x0 & ~x1) | (x1 & ~x0)
✓ Simplified Boolean: (x0 & ~x1) | (x1 & ~x0)
✓ XOR pairs detected: 1
  XOR: 10 ⊕ 01 at positions [0, 1]
✓ Different control sets: False
✓ Optimization: CX_trick

In [4]:
# Example 2: Partially Overlapping Control Patterns  
theta = np.pi / 4
qc = QuantumCircuit(3)
rx1 = multi_crx(theta, '11')
rx2 = multi_crx(theta, '10')
qc.append(rx1, [0, 1, 2])
qc.append(rx2, [0, 1, 2])


result2 = run_example_with_direct_transpilation(qc, "Partially Overlapping Control Patterns ['11', '10']")



EXAMPLE: Partially Overlapping Control Patterns ['11', '10']

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────o─────
     ┌────┴────┐┌────┴────┐
q_2: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Direct-Transpile) - CX: 16, U3: 15
Transpiled circuit depth: 25
Transpiled gate types: ['cx', 'u', 'u3', 'x', 'p']

--- Optimization Analysis ---
Optimization potential: 0.0%
XOR pairs found: 0
Can optimize: True

--- Applying Optimization ---
🔬 Starting  pattern reading MCRX simplification...
✓ Circuit validated: target=2, angle=0.7854, controls=2
✓ Extracted 2 patterns:
  ControlPattern('11', coeff=1, qubits=[0, 1])
  ControlPattern('10', coeff=1, qubits=[0, 1])
✓ Original Boolean: (x0 & x1) | (x0 & ~x1)
✓ Simplified Boolean: x0
✓ XOR pairs detected: 0
✓ Different control sets: False
✓ Optimization: single_control
✓ Gate reduction: 1
✓ Optimization complete

In [5]:
# Example 3: Complex Overlapping (3-qubit controls)

theta = 2 * np.pi / 4  # π/2
qc = QuantumCircuit(4)
rx1 = multi_crx(theta, '110')
rx2 = multi_crx(theta, '101')
qc.append(rx1, [0, 1, 2, 3])
qc.append(rx2, [0, 1, 2, 3])


result3 = run_example_with_direct_transpilation(qc, "Complex Overlapping Control Patterns ['110', '101']")


EXAMPLE: Complex Overlapping Control Patterns ['110', '101']

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────o─────
          │          │     
q_2: ─────o──────────■─────
     ┌────┴────┐┌────┴────┐
q_3: ┤ Rx(π/2) ├┤ Rx(π/2) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Direct-Transpile) - CX: 40, U3: 33
Transpiled circuit depth: 57
Transpiled gate types: ['cx', 'u', 'u3', 'x', 'p']

--- Optimization Analysis ---
Optimization potential: 0.0%
XOR pairs found: 1
Can optimize: True

--- Applying Optimization ---
🔬 Starting  pattern reading MCRX simplification...
✓ Circuit validated: target=3, angle=1.5708, controls=3
✓ Extracted 2 patterns:
  ControlPattern('110', coeff=1, qubits=[0, 1, 2])
  ControlPattern('101', coeff=1, qubits=[0, 1, 2])
✓ Original Boolean: (x0 & x1 & ~x2) | (x0 & x2 & ~x1)
✓ Simplified Boolean: x0 & (x1 | x2) & (~x1 | ~x2)
✓ XOR pairs detected: 1
  XOR

In [6]:
# Example 4: Subset Control Patterns
theta = np.pi / 4
qc = QuantumCircuit(3)
rx1 = multi_crx(theta, '11')
qc.append(rx1, [0, 1, 2])
# Fixed: Use MCRX instead of CRX to maintain same target requirement
rx2 = multi_crx(theta, '1')  # This is effectively a '1X' pattern where X can be 0 or 1
qc.append(rx2, [0, 2])  # Only qubit 0 as control, same target qubit 2

result4 = run_example_with_direct_transpilation(qc, "Subset Control Patterns ['11'] + MCRX('1', 0->2)")


EXAMPLE: Subset Control Patterns ['11'] + MCRX('1', 0->2)

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────┼─────
     ┌────┴────┐┌────┴────┐
q_2: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Direct-Transpile) - CX: 8, U3: 10
Transpiled circuit depth: 14
Transpiled gate types: ['cx', 'u3', 'u', 'p']

--- Optimization Analysis ---
Optimization potential: 0.0%
XOR pairs found: 0
Can optimize: False

--- Applying Optimization ---
🔬 Starting  pattern reading MCRX simplification...
✓ Circuit validated: target=2, angle=0.7854, controls=2
✓ Extracted 2 patterns:
  ControlPattern('11', coeff=1, qubits=[0, 1])
  ControlPattern('1', coeff=1, qubits=[0])
✓ Original Boolean: x0 | (x0 & x1)
✓ Simplified Boolean: x0
✓ XOR pairs detected: 0
✓ Different control sets: True
✓ Optimization: no_optimization_different_controls
✓ Blocked reason: Patterns have different co

In [7]:
# Example 5: Identical Control Patterns

theta = np.pi / 4
qc = QuantumCircuit(4)
rx1 = multi_crx(theta, '110')
rx2 = multi_crx(theta, '110')
qc.append(rx1, [0, 1, 2, 3])
qc.append(rx2, [0, 1, 2, 3])

result5 = run_example_with_direct_transpilation(qc, "Identical Control Patterns ['110', '110']")


EXAMPLE: Identical Control Patterns ['110', '110']

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────■─────
          │          │     
q_2: ─────o──────────o─────
     ┌────┴────┐┌────┴────┐
q_3: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Direct-Transpile) - CX: 40, U3: 31
Transpiled circuit depth: 57
Transpiled gate types: ['cx', 'u', 'u3', 'x', 'p']

--- Optimization Analysis ---
Optimization potential: 0.0%
XOR pairs found: 0
Can optimize: True

--- Applying Optimization ---
🔬 Starting  pattern reading MCRX simplification...
✓ Circuit validated: target=3, angle=0.7854, controls=3
✓ Extracted 1 patterns:
  ControlPattern('110', coeff=2, qubits=[0, 1, 2])
✓ Original Boolean: x0 & x1 & ~x2
✓ Simplified Boolean: x0 & x1 & ~x2
✓ XOR pairs detected: 0
✓ Different control sets: False
✓ Optimization: identical_patterns
✓ Gate reduction: 1
✓ Optimization